# EMG — comb filters compared by their envelopes

Loads **only** `emg_line_L.csv` from a folder you pick (same as
`emg_mains_filter_selector.ipynb`), then:

1. **Bandpass 5–500 Hz** — Butterworth, zero-phase `sosfiltfilt` (same reference).
2. Apply **both combs** to that reference:
   * **FIR comb** — feed-forward `y[n] = x[n] − x[n−M]`  (order 1)
   * **IIR comb** — feedback `H(z) = (1 − z⁻ᴹ)/(1 − r·z⁻ᴹ)`  (order 5, r = 0.9)
3. Plot three stacked panels: the **raw signal** on top (its own scale — it
   still carries the DC offset the bandpass removes), then each comb output's
   **amplitude envelope drawn over its filtered waveform** (mirrored to wrap the
   signal). The **two comb panels share one identical y-scale**, so the filters
   are directly comparable.

**Envelope** here means the EMG amplitude envelope, chosen by `ENVELOPE_METHOD`:

| value | how | notes |
|---|---|---|
| `"linear"` *(default)* | full-wave rectify → low-pass `ENV_CUTOFF_HZ` | classic EMG linear envelope |
| `"hilbert"` | `\|analytic signal\|`, then the same low-pass | no window choice needed |
| `"rms"` | moving-RMS over `ENV_WIN_MS` | window-length dependent |

Run top to bottom, then use the widget.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")
CHANNEL_FILE   = "emg_line_L.csv"   # this notebook reads this file and nothing else

# ── bandpass (the shared reference) ───────────────────────────────────────────
BAND      = (5.0, 500.0)     # Hz — EMG band / bandpass passband
BP_ORDER  = 4                # Butterworth order

# ── comb filters ──────────────────────────────────────────────────────────────
F0_MAINS  = 49.97328         # Hz — measured mains fundamental, sets the delay M
R_M       = 0.9              # IIR comb feedback coefficient
FIR_ORDER = 1                # feed-forward comb passes
IIR_ORDER = 5                # feedback comb cascade order

# ══ Envelope variables ════════════════════════════════════════════════════════
ENVELOPE_METHOD = "linear"   # "linear" (rectify+lowpass) | "hilbert" | "rms"
ENV_CUTOFF_HZ   = 5.0        # low-pass cutoff for "linear" / "hilbert" smoothing
ENV_WIN_MS      = 50.0       # moving-RMS window length for "rms\"

## Load & bandpass — identical to the selector notebook

`fs` comes from the median timestamp spacing (immune to one glitched row); the
bandpass is second-order sections because 5 Hz at 44.1 kHz is too low a
normalised frequency for stable `(b, a)` coefficients.

In [ ]:
def find_folders(root: Path = RECORDINGS_DIR):
    """Every folder under *root* holding an emg_line_L.csv (searched recursively)."""
    if not root.exists():
        return []
    return sorted(p.parent.relative_to(root).as_posix()
                  for p in root.rglob(CHANNEL_FILE))


def load_line_L(folder: str):
    """Read emg_line_L.csv from *folder* → (time, signal, fs)."""
    path = RECORDINGS_DIR / folder / CHANNEL_FILE
    if not path.exists():
        raise FileNotFoundError(f"{CHANNEL_FILE} not found in {path.parent}")
    df = pd.read_csv(path)
    t = df["time"].to_numpy(dtype=float)
    x = df["data"].to_numpy(dtype=float)
    fs = 1.0 / float(np.median(np.diff(t)))
    return t, x, fs


def bandpass_filter(x, fs, low_hz=BAND[0], high_hz=BAND[1], order=BP_ORDER):
    """Zero-phase Butterworth bandpass, SOS form for numerical stability."""
    nyq = fs / 2.0
    if high_hz >= nyq:
        high_hz = nyq * 0.99
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)


def rms(x):
    return float(np.sqrt(np.mean(np.asarray(x, float) ** 2)))

## The two combs

Ported unchanged from `emg_mains_filter_selector.ipynb`. Each is applied
zero-phase (`filtfilt`); an order-N cascade is N repeated passes.
`normalize=True` scales the numerator so one section peaks at exactly 1.0, so the
filter never inflates the signal.

In [ ]:
def comb_section(fs, kind, f0=F0_MAINS, r_m=R_M, normalize=True):
    """(b, a) for ONE comb section. kind is "ff" (FIR feed-forward) or "iir"."""
    M = int(round(fs / f0))
    b = np.zeros(M + 1)
    b[0], b[M] = 1.0, -1.0
    if kind == "ff":
        a = np.array([1.0])
        if normalize:
            b = b * 0.5
    else:
        a = np.zeros(M + 1)
        a[0], a[M] = 1.0, -r_m
        if normalize:
            b = b * (1.0 + r_m) / 2.0
    return b, a


def comb_filter(x, fs, kind, order=1, f0=F0_MAINS, r_m=R_M, normalize=True):
    """Apply a comb of the given cascade order, zero-phase."""
    b, a = comb_section(fs, kind, f0, r_m, normalize)
    y = x.astype(float)
    for _ in range(order):
        y = sig.filtfilt(b, a, y)
    return y

## The envelope

`signal_envelope` returns the amplitude envelope by the method in
`ENVELOPE_METHOD`. All three are non-negative, so a shared `[0, ymax]` y-axis is
the natural common scale for the two panels.

* **linear** — full-wave rectify (`|x|`) then a zero-phase Butterworth low-pass
  at `ENV_CUTOFF_HZ`. The textbook EMG *linear envelope*.
* **hilbert** — magnitude of the analytic signal `|x + j·H{x}|` (instantaneous
  amplitude), smoothed with the same low-pass so the line is readable.
* **rms** — square, moving-average over an `ENV_WIN_MS` window, square-root.

In [ ]:
def signal_envelope(x, fs, method=None, cutoff_hz=None, win_ms=None):
    """Amplitude envelope of *x*. Defaults fall back to the notebook constants."""
    method    = (method    or ENVELOPE_METHOD).lower()
    cutoff_hz =  cutoff_hz if cutoff_hz is not None else ENV_CUTOFF_HZ
    win_ms    =  win_ms    if win_ms    is not None else ENV_WIN_MS
    x = np.asarray(x, float)

    def lowpass(y):
        sos = sig.butter(4, cutoff_hz, btype="low", fs=fs, output="sos")
        return sig.sosfiltfilt(sos, y)

    if method == "linear":
        return np.clip(lowpass(np.abs(x)), 0.0, None)
    if method == "hilbert":
        env = np.abs(sig.hilbert(x))
        return np.clip(lowpass(env) if cutoff_hz else env, 0.0, None)
    if method == "rms":
        w = max(1, int(round(win_ms * 1e-3 * fs)))
        kernel = np.ones(w) / w
        return np.sqrt(np.convolve(x ** 2, kernel, mode="same"))
    raise ValueError(f'ENVELOPE_METHOD must be "linear", "hilbert" or "rms", got {method!r}')


def minmax_decimate(t, x, n_bins=6000):
    """Peak-preserving decimation for plotting: keep the real min & max per bin."""
    n = len(x)
    if n <= 2 * n_bins:
        return t, x
    bin_len = n // n_bins
    usable = bin_len * n_bins
    idx = np.arange(usable).reshape(n_bins, bin_len)
    xb = x[:usable].reshape(n_bins, bin_len)
    rows = np.arange(n_bins)
    keep = np.union1d(idx[rows, np.argmin(xb, axis=1)],
                      idx[rows, np.argmax(xb, axis=1)])
    if usable < n:
        keep = np.union1d(keep, np.arange(usable, n))
    return t[keep], x[keep]

## Main — bandpass → both combs → envelope over the filtered signal (shared scale)

In [ ]:
RAW_COLOR = "#1565C0"                 # blue  — raw signal (top panel)
FIR_COLOR = "#2E7D32"                 # green — FIR feed-forward comb envelope
IIR_COLOR = "#8E0000"                 # red   — IIR feedback comb envelope
SIG_COLOR = "rgba(120,120,120,0.45)"  # faded grey — the filtered waveform behind


def analyze(folder, low_hz=BAND[0], high_hz=BAND[1], bp_order=BP_ORDER,
            env_method=None, env_cutoff=None, env_win_ms=None):
    """Bandpass emg_line_L.csv, apply both combs, overlay each envelope on its
    filtered signal — two panels on one shared, symmetric y-scale."""

    env_method = (env_method or ENVELOPE_METHOD).lower()
    t, raw, fs = load_line_L(folder)
    bp = bandpass_filter(raw, fs, low_hz, high_hz, bp_order)

    fir = comb_filter(bp, fs, "ff",  order=FIR_ORDER, normalize=True)
    iir = comb_filter(bp, fs, "iir", order=IIR_ORDER, r_m=R_M, normalize=True)

    env_fir = signal_envelope(fir, fs, env_method, env_cutoff, env_win_ms)
    env_iir = signal_envelope(iir, fs, env_method, env_cutoff, env_win_ms)

    fir_lbl = f"FIR feed-forward comb  (order {FIR_ORDER})"
    iir_lbl = f"IIR feedback comb  (order {IIR_ORDER}, r={R_M})"

    # ── summary ───────────────────────────────────────────────────────────────
    print(f"{folder}/{CHANNEL_FILE}")
    print(f"  fs = {fs:,.1f} Hz   {len(raw):,} samples   {t[-1] - t[0]:.3f} s")
    print(f"  bandpass {low_hz:g}–{high_hz:g} Hz  →  FIR comb & IIR comb")
    cut = env_cutoff if env_cutoff is not None else ENV_CUTOFF_HZ
    win = env_win_ms if env_win_ms is not None else ENV_WIN_MS
    detail = (f"cutoff {cut:g} Hz" if env_method in ("linear", "hilbert")
              else f"window {win:g} ms")
    print(f"  envelope: method={env_method!r}  ({detail})")
    print(f"  envelope peak   FIR = {env_fir.max():.4g} V    IIR = {env_iir.max():.4g} V")
    print(f"  envelope RMS    FIR = {rms(env_fir):.4g} V    IIR = {rms(env_iir):.4g} V")

    # ── one common symmetric scale for both panels ────────────────────────────
    # The waveform is bipolar and its peaks exceed the (smoothed) envelope, so the
    # range must cover the signal itself, not just the envelope, to avoid clipping.
    ymax = float(max(np.max(np.abs(fir)), np.max(np.abs(iir)),
                     env_fir.max(), env_iir.max())) * 1.05

    # shared_yaxes is left OFF so the raw panel keeps its own scale (it still has
    # the DC offset the bandpass removes); the two comb panels are pinned to the
    # SAME explicit [-ymax, ymax] range below, which is what "same scale" means.
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        vertical_spacing=0.08,
                        subplot_titles=["Raw signal",
                                        f"{fir_lbl} — signal + envelope",
                                        f"{iir_lbl} — signal + envelope"])

    # row 1 — raw signal, on its own auto-scaled axis
    tr, yr = minmax_decimate(t, raw)
    fig.add_trace(go.Scattergl(x=tr, y=yr, name="Raw",
                               line=dict(width=0.6, color=RAW_COLOR)), row=1, col=1)
    fig.update_yaxes(title_text="Amplitude (V)", row=1, col=1)

    # rows 2-3 — each comb's filtered waveform with its envelope drawn over it
    for r, (y, env, name, color) in enumerate(
            [(fir, env_fir, "FIR comb", FIR_COLOR),
             (iir, env_iir, "IIR comb", IIR_COLOR)], start=2):
        # 1) the comb-filtered waveform, faded, in the background
        ts, ys = minmax_decimate(t, y)
        fig.add_trace(go.Scattergl(x=ts, y=ys, name=f"{name} (filtered)",
                                   legendgroup=name + "_sig",
                                   line=dict(width=0.5, color=SIG_COLOR)),
                      row=r, col=1)
        # 2) the envelope ON TOP, plus its mirror, so it clearly wraps the signal
        te, ye = minmax_decimate(t, env)
        fig.add_trace(go.Scattergl(x=te, y=ye, name=f"{name} envelope",
                                   legendgroup=name + "_env",
                                   line=dict(width=1.8, color=color)),
                      row=r, col=1)
        fig.add_trace(go.Scattergl(x=te, y=-ye, name=f"{name} envelope",
                                   legendgroup=name + "_env", showlegend=False,
                                   line=dict(width=1.8, color=color)),
                      row=r, col=1)
        # force the SAME scale on both comb panels (not just shared zoom)
        fig.update_yaxes(title_text="Amplitude (V)", range=[-ymax, ymax], row=r, col=1)

    fig.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig.update_layout(
        title_text=(f"{folder} — raw + comb-filtered signals & envelope "
                    f"(method: {env_method}) — combs share y-scale"),
        height=900, template="plotly_white", hovermode="x unified",
        legend=dict(orientation="h", y=1.05, x=0.5, xanchor="center"))
    fig.show()

    print("\nDone.")
    return dict(t=t, fir=fir, iir=iir, env_fir=env_fir, env_iir=env_iir, fs=fs)

## Widget — choose a folder and run

Widgets are built **once** and reuse a **single** output area (the
`try/except NameError` guard), so re-running the cell never leaves duplicate
buttons or two stacked copies of the figure.

In [ ]:
clear_output(wait=True)

folders = find_folders()

if not folders:
    print(f"No folder under '{RECORDINGS_DIR}/' contains {CHANNEL_FILE}.")
else:
    def _num(w_cls, value, desc, width="220px", **kw):
        return w_cls(value=value, description=desc,
                     style={"description_width": "initial"},
                     layout=widgets.Layout(width=width), **kw)

    try:
        _ev_run                                  # already built on an earlier run?
    except NameError:
        _ev_folder = widgets.Dropdown(
            options=folders, value=folders[0], description="Recording:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="520px"))
        _ev_low    = _num(widgets.BoundedFloatText, 5.0,   "Low cutoff (Hz):", "210px",
                          min=0.1, max=2000.0, step=1.0)
        _ev_high   = _num(widgets.BoundedFloatText, 500.0, "High cutoff (Hz):", "210px",
                          min=1.0, max=20000.0, step=10.0)
        _ev_bpord  = _num(widgets.BoundedIntText,   4,     "BP order:", "150px",
                          min=1, max=8, step=1)
        _ev_method = widgets.Dropdown(
            options=[("linear (rectify+lowpass)", "linear"),
                     ("hilbert (|analytic|)",     "hilbert"),
                     ("rms (moving window)",      "rms")],
            value="linear", description="Envelope:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="320px"))
        _ev_cut    = _num(widgets.BoundedFloatText, 5.0,  "Env cutoff (Hz):", "210px",
                          min=0.5, max=50.0, step=0.5)
        _ev_win    = _num(widgets.BoundedFloatText, 50.0, "RMS window (ms):", "210px",
                          min=1.0, max=1000.0, step=5.0)
        _ev_run    = widgets.Button(description="▶  Bandpass, comb & envelope",
                                    button_style="primary",
                                    layout=widgets.Layout(width="260px", height="38px"))
        _ev_out    = widgets.Output()
    else:
        _ev_folder.options = folders
        if _ev_folder.value not in folders:
            _ev_folder.value = folders[0]

    def _on_run(b):
        with _ev_out:
            clear_output(wait=True)
            try:
                analyze(_ev_folder.value,
                        low_hz=float(_ev_low.value), high_hz=float(_ev_high.value),
                        bp_order=int(_ev_bpord.value),
                        env_method=_ev_method.value,
                        env_cutoff=float(_ev_cut.value),
                        env_win_ms=float(_ev_win.value))
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")

    _ev_run._click_handlers.callbacks[:] = []   # exactly one handler
    _ev_run.on_click(_on_run)

    display(widgets.VBox([
        widgets.HTML(f"<b>Pick a recording — only <code>{CHANNEL_FILE}</code> is read.</b>"),
        _ev_folder,
        widgets.HBox([_ev_low, _ev_high, _ev_bpord]),
        widgets.HBox([_ev_method, _ev_cut, _ev_win]),
        _ev_run,
        _ev_out,
    ]))